# MOSS-Image on Kaggle (ComfyUI)

部署 ComfyUI（最新 master，含 MOSS-Image 原生支持）+ Moody Moss Mix V3.3（MOSS-Image 架构 INT8 checkpoint，从账号私有数据集 moody-moss-mix-v33 挂载读取，不再从 HuggingFace 下载）+ Qwen3.5-9B w4a8 + FLUX.2 small VAE，跑一张竖版 3:4 林间人像测试图。KSampler 25 步、CFG 3.5（作者推荐值）。

模型来自 `Comfy-Org/MOSS-Image`（公开仓库，Apache 2.0，无需 token）。测试提示词沿用之前在 SDXL 上被 CLIP 截断的那段长提示词，正好验证 MOSS-Image 约 4K token 的提示词容量；提示词增强器不开，提示词原样送入模型。

注意：ComfyUI 服务由出图那格用 subprocess.Popen 在本进程内启动（Kaggle 版本运行里跨格的 nohup 后台进程活不下来），跑完即停。

LoRA 预留：下载格和出图格里各有一段 LoRA 配置（LORA_NAME / LORA_REPO / LORA_STRENGTH，默认空）。需要加 LoRA 时填上文件名和仓库，下载格会把它下到 ComfyUI/models/loras/，出图格会在 UNETLoader 后接一个 LoraLoaderModelOnly 节点（只改模型，不动 CLIP）；留空时节点不接入，模型线完全不变。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working
!python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
!if [ -d /kaggle/working/ComfyUI ]; then echo comfyui-exists; else git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI; fi
!cd /kaggle/working/ComfyUI && grep -v -E "^(torch|torchvision|torchaudio)(==|>=|~=|>|<| |$)" requirements.txt > /tmp/comfy_reqs.txt && pip install -q -r /tmp/comfy_reqs.txt
!pip install -q torchsde
!python3 -c "import torchsde; print('torchsde ok')"
!ls /kaggle/working/ComfyUI | head -20

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
from huggingface_hub import hf_hub_download
base = "/kaggle/working/ComfyUI/models"
# NOTE: filenames must include the repo subdirectory; hf_hub_download places the file at local_dir/filename
#
# LoRA (optional, reserved for later use): set LORA_NAME to a .safetensors file
# in ComfyUI/models/loras/ and LORA_REPO to the Hugging Face repo hosting it
# (file assumed at the repo root). Leave LORA_NAME = "" (default) to run
# without a LoRA. When set, the file is downloaded here and the run cell wires
# a LoraLoaderModelOnly node into the model line.
LORA_NAME = ""
LORA_REPO = ""
LORA_STRENGTH = 1.0
jobs = [
    "text_encoders/qwen3.5_9b_w4a8_gptq.safetensors",
    "vae/flux2-vae_small.safetensors",
]
for fname in jobs:
    p = hf_hub_download(repo_id="Comfy-Org/MOSS-Image", filename=fname, local_dir=base)
    print(fname, round(os.path.getsize(p) / 1e9, 2), "GB")
# Main diffusion model (2026-11-12): Moody Moss Mix V3.3, an INT8
# MOSS-Image checkpoint hosted on Civitai. It is NOT downloaded from
# HuggingFace anymore: it reaches Kaggle as the private dataset
# moody-moss-mix-v33 attached to this kernel, and is located by a
# recursive search under /kaggle/input (the real mount path varies,
# never hard-code it), then copied into ComfyUI's diffusion_models.
import glob, shutil
MOODY_NAME = "Moody-Moss-Mix-V3.3_00001__clean-int8_convrot.safetensors"
hits = sorted(glob.glob("/kaggle/input/**/" + MOODY_NAME, recursive=True))
if not hits:
    raise SystemExit("moody model not found under /kaggle/input: dataset moody-moss-mix-v33 is not attached to this kernel")
moody_dst = os.path.join(base, "diffusion_models", MOODY_NAME)
os.makedirs(os.path.dirname(moody_dst), exist_ok=True)
shutil.copyfile(hits[0], moody_dst)
print("moody model:", hits[0], "->", moody_dst, round(os.path.getsize(moody_dst) / 1e9, 2), "GB")
if LORA_NAME:
    p = hf_hub_download(repo_id=LORA_REPO, filename=LORA_NAME, local_dir=base + "/loras")
    print("lora", LORA_NAME, round(os.path.getsize(p) / 1e9, 2), "GB")
print("all models downloaded")

In [ ]:
import json, time, requests, os, subprocess, sys

BASE = "http://127.0.0.1:8188"
LOG = "/kaggle/working/comfy_server.log"

# Start the ComfyUI server as a child of this cell's python process (nohup across cells does not survive Kaggle version runs).
logf = open(LOG, "wb")
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd="/kaggle/working/ComfyUI",
    stdout=logf, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("server pid:", proc.pid)

up = False
for i in range(36):
    if proc.poll() is not None:
        print("server exited early, code:", proc.returncode)
        break
    try:
        r = requests.get(BASE + "/system_stats", timeout=5)
        if r.status_code == 200:
            up = True
            print("server up after", (i + 1) * 5, "s")
            break
    except Exception:
        pass
    time.sleep(5)
if not up:
    print("server did not come up; log tail:")
    print(open(LOG, errors="ignore").read()[-3000:])
    proc.terminate()
    raise SystemExit("comfyui server failed to start")

PROMPT = """清透柔软的成年面容，配合轻薄低领穿搭的私密时尚写真。绝美23岁东亚美女，身材匀称丰满，胸部明显自然丰满，视觉约E杯，具有自然重量感与柔和圆润轮廓，胸腰差明显但自然，纤细腰部与丰满胸部形成清晰比例，臀腿曲线顺滑圆润饱满。自然散落的黑长发拨在背后，底妆细腻，眉眼舒展，红润唇色带一点柔光；脸微偏向镜头，目光柔软而专注。
她穿香槟色薄绉纱短衬衣，前襟低开成深V，宽松柔软的领边自然落向两侧，深V两侧胸片采用同色不透明薄缎，肩后与短袖保留透光，前摆在腰线上方打一个小结。搭配黑色低腰斜裁微迷你裙，薄缎贴合腰髋，裙摆刚过臀下缘，臀部覆盖完整。她斜倚在窗边窄柜旁，一手撑住身后柜沿，肩颈放松，上身轻转向镜头，近侧腿向前半步，露出的腰线与裙身比例鲜明。
4:5竖幅，50mm取景至大腿中段。侧窗柔光让眼睛、肤色和绉纱边缘清楚透亮，远侧身体保留阴影，背景简洁柔暗。衣料细褶、透光短袖与自然肤理是质感重点。"""
NEGATIVE = "nsfw, worst quality, bad anatomy, watermark, jpeg artifact, blurry, low quality, distorted, deformed, extra limbs, text, lettering, wide landscape, decorative flowers, fantasy styling, cold porcelain skin, excessive skin smoothing, studio light, exaggerated anatomy, visible intimate anatomy"

info = requests.get(BASE + "/object_info", timeout=120).json()
have = set(info.keys())
# LoRA settings are defined in the download cell; default to "no LoRA" if that cell was skipped.
LORA_NAME = globals().get("LORA_NAME", "")
LORA_STRENGTH = globals().get("LORA_STRENGTH", 1.0)
need = ["UNETLoader", "CLIPLoader", "VAELoader", "CLIPTextEncode", "EmptyFlux2LatentImage", "KSampler", "VAEDecode", "SaveImage"]
if LORA_NAME:
    need.append("LoraLoaderModelOnly")
missing = [n for n in need if n not in have]
print("missing nodes:", missing if missing else "none")
if missing:
    proc.terminate()
    raise SystemExit("missing nodes: " + ", ".join(missing))

def node_inputs(class_type, preferred):
    spec = info[class_type]["input"]
    allowed = set(spec.get("required", {})) | set(spec.get("optional", {}))
    return {k: v for k, v in preferred.items() if k in allowed}

use_slg = "SkipLayerGuidanceDiTSimple" in have
print("SLG available:", use_slg)

nodes = {}
nodes["1"] = {"class_type": "UNETLoader", "inputs": node_inputs("UNETLoader", {"unet_name": "Moody-Moss-Mix-V3.3_00001__clean-int8_convrot.safetensors", "weight_dtype": "default"})}
nodes["2"] = {"class_type": "CLIPLoader", "inputs": node_inputs("CLIPLoader", {"clip_name": "qwen3.5_9b_w4a8_gptq.safetensors", "type": "moss_image", "device": "default"})}
nodes["3"] = {"class_type": "VAELoader", "inputs": node_inputs("VAELoader", {"vae_name": "flux2-vae_small.safetensors"})}
nodes["4"] = {"class_type": "CLIPTextEncode", "inputs": {"text": PROMPT, "clip": ["2", 0]}}
nodes["5"] = {"class_type": "CLIPTextEncode", "inputs": {"text": NEGATIVE, "clip": ["2", 0]}}
nodes["6"] = {"class_type": "EmptyFlux2LatentImage", "inputs": node_inputs("EmptyFlux2LatentImage", {"width": 1696, "height": 2112, "batch_size": 1})}
print("latent node spec:", json.dumps(info.get("EmptyFlux2LatentImage", {}).get("input", {}))[:600])
print("latent node built:", json.dumps(nodes["6"]))
model_link = ["1", 0]
# LoRA (optional): when LORA_NAME is set, patch the model with LoraLoaderModelOnly
# (model-only, no CLIP) before anything else consumes it. When empty, the model
# line is unchanged (UNETLoader straight to SLG/KSampler).
if LORA_NAME:
    nodes["20"] = {"class_type": "LoraLoaderModelOnly", "inputs": node_inputs("LoraLoaderModelOnly", {"model": ["1", 0], "lora_name": LORA_NAME, "strength_model": LORA_STRENGTH})}
    model_link = ["20", 0]
if use_slg:
    nodes["8"] = {"class_type": "SkipLayerGuidanceDiTSimple", "inputs": node_inputs("SkipLayerGuidanceDiTSimple", {"model": model_link, "single_layers": "22", "double_layers": "", "scale": 3.0, "start_percent": 0.01, "end_percent": 1.0})}
    model_link = ["8", 0]
nodes["7"] = {"class_type": "KSampler", "inputs": {"model": model_link, "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0], "seed": 42, "steps": 25, "cfg": 3.5, "sampler_name": "er_sde", "scheduler": "sgm_uniform", "denoise": 1.0}}
nodes["9"] = {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}}
nodes["10"] = {"class_type": "SaveImage", "inputs": {"images": ["9", 0], "filename_prefix": "moss_fashion"}}

# Pre-check: every required input of every node must be present, or fail here with the full list (do not let the server reject it one input at a time).
problems = []
for nid, node in nodes.items():
    req = info[node["class_type"]]["input"].get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems.append(nid + ":" + node["class_type"] + "." + name)
print("missing required inputs:", problems if problems else "none")
if problems:
    proc.terminate()
    raise SystemExit("missing required inputs: " + ", ".join(problems))

r = requests.post(BASE + "/prompt", json={"prompt": nodes}, timeout=180)
print("submit:", r.status_code, r.text[:400])
r.raise_for_status()
pid = r.json()["prompt_id"]
print("prompt_id:", pid)

out_path = "/kaggle/working/moss_fashion.png"
t0 = time.time()
done = False
while time.time() - t0 < 3300:
    h = requests.get(BASE + "/history/" + pid, timeout=60).json()
    if pid in h:
        st = h[pid].get("status", {})
        sstr = st.get("status_str")
        if sstr == "success":
            outs = h[pid]["outputs"]
            imgs = outs.get("10", {}).get("images", [])
            print("images:", imgs)
            if imgs:
                im = imgs[0]
                q = {"filename": im["filename"], "subfolder": im.get("subfolder", ""), "type": im.get("type", "output")}
                data = requests.get(BASE + "/view", params=q, timeout=300).content
                open(out_path, "wb").write(data)
                print("saved", out_path, len(data), "bytes")
                done = True
            break
        if sstr == "error":
            print("RUN ERROR:", json.dumps(h[pid], indent=2)[:3000])
            break
    time.sleep(15)
print("done:", done, "elapsed:", round(time.time() - t0), "s")
proc.terminate()
print("server stopped")

In [ ]:
from IPython.display import Image, display
import os
p = "/kaggle/working/moss_fashion.png"
print("exists:", os.path.exists(p), "bytes:", os.path.getsize(p) if os.path.exists(p) else 0)
display(Image(filename=p))